# Lab 03 (Colab) - Sampling Knobs and Post-Training

**Chapter 2 (Understanding Foundation Models): Post-Training and Sampling** | Environment: **Google Colab, free T4 GPU** | Time: about 1.5 hours (do this notebook before `lab03_cluster.ipynb`)

A model does not output an answer; it outputs a probability for every token in its vocabulary, and a **sampling** strategy picks one. Chapter 2 argues that this one step explains a lot of model behavior: creativity, inconsistency, even hallucination. Every model API you will use at work, including the course cluster, exposes the same few sampling settings: `temperature`, `top_k`, and `top_p`. In this notebook you turn those knobs on a real model, watch what each one does to the next-token probabilities and to real answers, and choose the settings you would ship for two Larkspur jobs. Then you measure how temperature changes the diversity and consistency of answers to Larkspur questions. Finally you compare a **base** model with its **post-trained** (instruct) sibling to see what post-training adds.

Models: `Qwen/Qwen3-1.7B` (post-trained) and `Qwen/Qwen3-1.7B-Base` (pretrained only). Both fit on a T4 at the same time (about 3.5 GB each in float16).

**Before you start**
- **Runtime > Change runtime type > T4 GPU > Save.**
- Model downloads are about 4 GB each.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Logit** | The raw score the model gives each token in its vocabulary for "what comes next" (from Lab 01). Higher means more likely. | Every sampling method in this lab works on these scores. |
| **Softmax** | The formula that turns logits into probabilities that add up to 100% (from Lab 01). | `torch.softmax` does it for you in A.1. |
| **Probability distribution** | The full list of probabilities, one per token, for the next token. | "Sharp" (one token very likely) vs "flat" (many tokens similar) is what temperature changes. |
| **Sampling** | Picking the next token at random, weighted by its probability. | It's why a model can give a different answer each time you ask. |
| **Greedy decoding** | Always picking the single most likely token instead of sampling (from Lab 01). | Same input, same output (in theory; the cluster notebook tests it). |
| **Temperature** | A setting that sharpens (below 1) or flattens (above 1) the distribution before sampling. | The main knob for "creative" vs "consistent" output. |
| **Top-k / top-p** | Filters that throw away unlikely tokens before sampling: keep the `k` best, or keep the smallest set that adds up to probability `p`. | They stop the model from occasionally picking a nonsense token. |
| **Survivors** | This lab's word for the tokens still allowed after top-k or top-p filtering. | A.3 counts them for two prompts. |
| **Entropy** | A number (in bits) for how spread out a distribution is. 0 bits means certain; higher means less sure. | You use it to measure how confident a model is. |
| **Tensor** | PyTorch's name for a list or grid of numbers that can live on the GPU. | Logits and token IDs are tensors in the code. |
| **n-gram, distinct-n** | An n-gram is a run of n words in a row. Distinct-n is the share of n-grams that are unique across several answers. | It measures how varied a batch of answers is (B.1). |
| **Hallucination** | A confident answer that is made up. | Part B shows when it happens and whether temperature affects it. |
| **Grounded** | The model was given the facts it needs in its context (for example, a document). | Grounding is the main defense against hallucination. |
| **Base vs post-trained (instruct) model** | A base model only learned to continue text. A post-trained model was then trained further to follow instructions and chat (from Lab 01). | B.3 compares the two directly. |
| **SFT, preference finetuning** | The two post-training steps in Chapter 2: supervised finetuning on example conversations, then training on which answers people prefer (RLHF, DPO). | They explain the differences you see in B.3. |

## Setup

The first cell installs the course helper package and an up-to-date `transformers` (Hugging Face's library for running open models; `accelerate` helps it place the model on the GPU). The second cell loads the instruct model and defines three helpers you'll use all the way through:
- `chat_inputs()` turns chat messages into token IDs, using the model's **chat template** (the special-token format from Lab 01 that marks the system, user, and assistant turns). It also turns off Qwen3's "thinking" mode, where the model writes out its reasoning before answering.
- `next_logits()` runs the model once and returns its scores (logits) for the first token of the answer.
- `show_top()` prints the most likely next tokens as a small table.

In PyTorch, numbers live in **tensors**: lists or grids of numbers that can sit on the GPU.

In [ ]:
!pip -q install "itec632 @ git+https://github.com/tyler-schunk/itec632-course#subdirectory=image"
!pip -q install -U transformers accelerate

In [ ]:
import re, math, time
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from itec632.grading import LabChecker

ck = LabChecker("lab03", part="colab")
gpu = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if gpu else "NONE - use Runtime > Change runtime type > T4 GPU")
ck.check("setup_gpu", gpu, 1, "Select a T4 GPU (Runtime > Change runtime type), then Runtime > Run all.")

INSTRUCT_ID = "Qwen/Qwen3-1.7B"
DTYPE = torch.float16   # if you see "nan"/"inf" errors, change to torch.float32 (the T4 has no bfloat16 support).
                        # float32 doubles memory: if Colab runs out of memory at B.3, Runtime > Restart session, Run all.
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_ID)
model = AutoModelForCausalLM.from_pretrained(INSTRUCT_ID, dtype=DTYPE, device_map="auto")
model.eval()
print("loaded", INSTRUCT_ID, "on", model.device)


def chat_inputs(messages, tok=None, mdl=None):
    """Chat messages -> tensors. enable_thinking=False turns off Qwen3's reasoning phase for this lab."""
    tok, mdl = tok or tokenizer, mdl or model
    return tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                   return_tensors="pt", return_dict=True).to(mdl.device)


def next_logits(messages):
    """Raw logits (float32, 1-D over the vocabulary) for the first answer token."""
    with torch.no_grad():
        return model(**chat_inputs(messages)).logits[0, -1, :].float()


def show_top(logits, k=8, title=""):
    probs = torch.softmax(logits, dim=-1)
    top_p, top_i = torch.topk(probs, k)
    rows = [{"token": repr(tokenizer.decode([int(i)])), "logit": round(logits[i].item(), 2),
             "prob_%": round(p.item() * 100, 2)} for p, i in zip(top_p, top_i)]
    if title:
        print(title)
    display(pd.DataFrame(rows))

# Part A - Guided practice: turn the knobs

## A.1 From logits to probabilities

For each position the model produces one raw score, a **logit**, per vocabulary token. Chapter 2 (**"Sampling Fundamentals"**) explains that **softmax** turns logits into probabilities that add up to 1. You don't write softmax yourself: PyTorch's `torch.softmax` does it in one call.

The prompt below is a help-desk style question with many reasonable one-word answers, so the probability is spread over several tokens. That makes the effect of each knob easy to see.

**Reading the output:** `vocabulary size` is how many tokens the model scores at every step. The table lists the 8 most likely first tokens of the answer: `token` is the piece of text (in quotes so you can see spaces), `logit` is its raw score, and `prob_%` is its probability after softmax.

In [ ]:
PROMPT = [{"role": "system", "content": "You are Larkspur Ridge IT's help-desk assistant."},
          {"role": "user", "content": "Give one word that describes a great IT technician. Reply with just the word."}]
logits = next_logits(PROMPT)             # one raw score per vocabulary token
probs = torch.softmax(logits, dim=-1)    # the library turns the scores into probabilities
print("vocabulary size:", logits.shape[0])
print(f"probabilities add up to {probs.sum().item():.4f}")
show_top(logits, title="Top candidates at temperature 1 (no filters):")

**TODO A.1:** read the table. Type the most likely token (without the quotes) and its probability in percent. You'll compare against these numbers in A.2.

In [ ]:
top_token = ""             # TODO: the most likely token from the first row of the table (no quotes needed)
top_token_prob_pct = None  # TODO: its prob_% from the table, as a number (for example 42.5)

In [ ]:
_top_id = int(torch.argmax(probs))
_typed = str(top_token).strip().strip("'\"").strip().lower()
try:
    _pct_ok = abs(float(top_token_prob_pct) - probs[_top_id].item() * 100) < 1.0
except (TypeError, ValueError):
    _pct_ok = False
ck.check("A1_read_top_token", _typed == tokenizer.decode([_top_id]).strip().lower() and _pct_ok, 2,
         "Copy the first row of the table: the token text (without the quotes) and its prob_% number.")
ck.record("top_token_prob_pct_T1", round(probs[_top_id].item() * 100, 2));

## A.2 Temperature (and greedy decoding)

Temperature divides every logit by `T` before softmax (Chapter 2, **"Sampling Strategies"**). `T < 1` sharpens the distribution (the favorite token gets even more likely), `T > 1` flattens it (rare tokens get a real chance). `T = 0` would divide by zero, so libraries treat it as "always take the most likely token" (**greedy decoding**): the same input gives the same output.

**Entropy** (in bits) summarizes how spread out a distribution is: 0 bits means certain, 1 bit is like a coin flip between two equally likely tokens, and each extra bit doubles the number of equally likely choices.

The next cell defines helpers. Run it; you don't need to change it.
- `apply_knobs(logits, temperature, top_k, top_p)` applies the three settings with Hugging Face's own logits processors, the same building blocks `generate()` uses, in the same order (temperature, then top-k, then top-p). Tokens that get filtered out receive a score of `-inf`, which means probability 0.
- `compare_knobs(logits, settings)` returns a table with one row per setting: how many tokens **survive** (`survivors`), the top token and its probability, and the entropy.
- `show_knobs(logits, ...)` draws a bar chart of the 10 most likely tokens before (grey) and after (blue) the settings.

In [ ]:
from transformers import TemperatureLogitsWarper, TopKLogitsWarper, TopPLogitsWarper


def entropy_bits(logits):
    """Entropy in bits of softmax(logits). 0 = certain; higher = more spread out."""
    p = torch.softmax(logits.float(), dim=-1)
    p = p[p > 0]
    return float(-(p * torch.log2(p)).sum())


def apply_knobs(logits, temperature=1.0, top_k=0, top_p=1.0):
    """Temperature, then top-k, then top-p, using transformers' logits processors.
    top_k=0 and top_p=1.0 mean 'filter off'. Removed tokens get -inf (probability 0)."""
    scores = logits.float().unsqueeze(0)                                  # processors expect a batch of rows
    ids = torch.zeros((1, 1), dtype=torch.long, device=scores.device)     # not used by these three processors
    if temperature != 1.0:
        scores = TemperatureLogitsWarper(temperature)(ids, scores)
    if top_k and top_k > 0:
        scores = TopKLogitsWarper(top_k)(ids, scores)
    if top_p < 1.0:
        scores = TopPLogitsWarper(top_p)(ids, scores)
    return scores[0]


def compare_knobs(logits, settings):
    """settings: a list of dicts with any of temperature / top_k / top_p. Returns one table row per setting."""
    rows = []
    for s in settings:
        out = apply_knobs(logits, **s)
        p = torch.softmax(out, dim=-1)
        top = int(torch.argmax(p))
        rows.append({"temperature": s.get("temperature", 1.0), "top_k": s.get("top_k", 0), "top_p": s.get("top_p", 1.0),
                     "survivors": int(torch.isfinite(out).sum()), "top1": tokenizer.decode([top]),
                     "top1_prob": round(p[top].item(), 3), "entropy_bits": round(entropy_bits(out), 2)})
    return pd.DataFrame(rows)


def show_knobs(logits, temperature=1.0, top_k=0, top_p=1.0, n=10):
    """Bar chart of the n most likely tokens: probability before (grey) and after (blue) the settings."""
    out = apply_knobs(logits, temperature, top_k, top_p)
    before, after = torch.softmax(logits.float(), dim=-1), torch.softmax(out, dim=-1)
    idx = torch.topk(before, n).indices
    labels = [repr(tokenizer.decode([int(i)])) for i in idx]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh([y + 0.2 for y in range(n)], before[idx].tolist(), height=0.4, color="lightgrey", label="T=1, no filters")
    ax.barh([y - 0.2 for y in range(n)], after[idx].tolist(), height=0.4, color="tab:blue", label="your settings")
    ax.set_yticks(list(range(n)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("probability")
    ax.legend(loc="lower right")
    ax.set_title(f"temperature={temperature}, top_k={top_k}, top_p={top_p}: "
                 f"{int(torch.isfinite(out).sum())} of {logits.numel()} tokens survive", fontsize=10)
    plt.tight_layout()
    plt.show()


show_knobs(logits)   # no settings yet: both bars are the same

**TODO A.2 (predict first, then choose):**
1. Before you run the table: at `T = 0.3`, will the top token's probability be *higher* or *lower* than the `prob_%` you read in A.1 (which was at `T = 1`)? Write your guess in `prediction_temperature`. It is not graded for being right; it is there so you notice whether your intuition matches what happens.
2. Choose at least four temperatures to compare in `MY_TEMPERATURES`. Include at least one below 1 and one above 1, and stay between 0.1 and 2.0.

In [ ]:
prediction_temperature = ""   # TODO: "higher" or "lower"
MY_TEMPERATURES = []          # TODO: at least four temperatures between 0.1 and 2.0 (some below 1, some above)

In [ ]:
temps = sorted(t for t in MY_TEMPERATURES if isinstance(t, (int, float)) and t > 0)
if len(temps) < 2:
    temp_table = None
    print("Fill in MY_TEMPERATURES in the cell above first (at least two temperatures above 0).")
else:
    temp_table = compare_knobs(logits, [{"temperature": t} for t in temps])
    display(temp_table)
    show_knobs(logits, temperature=temps[0])
    show_knobs(logits, temperature=temps[-1])

**Reading the table:** one row per temperature, lowest first. `survivors` is the whole vocabulary in every row: temperature reshapes the distribution but never removes a token. `top1` is the most likely token and `top1_prob` its probability (a fraction: 0.42 means 42%); `entropy_bits` is how spread out the whole distribution is. The two charts show your lowest and highest temperature. Was your prediction right?

**TODO A.2 (read off):** record the top token's probability at your lowest and your highest temperature, copied from the `top1_prob` column.

In [ ]:
top1_prob_lowest_T = None    # TODO: top1_prob in the first row of your table
top1_prob_highest_T = None   # TODO: top1_prob in the last row of your table

In [ ]:
ok_t = temp_table is not None
ck.check("A2_temperatures_chosen", ok_t and len(temps) >= 4 and min(temps) < 1 < max(temps) and max(temps) <= 2.0, 1,
         "MY_TEMPERATURES needs at least four values between 0.1 and 2.0, with at least one below 1 and one above 1.")
try:
    _read_ok = ok_t and abs(float(top1_prob_lowest_T) - temp_table["top1_prob"].iloc[0]) < 0.01 \
        and abs(float(top1_prob_highest_T) - temp_table["top1_prob"].iloc[-1]) < 0.01
except (TypeError, ValueError):
    _read_ok = False
ck.check("A2_read_values", _read_ok, 2,
         "Copy top1_prob from the first and the last row of your table (numbers between 0 and 1, not percentages).")
if ok_t:
    ck.record("temperatures_tried", temps)
    ck.record("top1_prob_lowest_T", float(temp_table["top1_prob"].iloc[0]))
    ck.record("top1_prob_highest_T", float(temp_table["top1_prob"].iloc[-1]))
    ck.record("entropy_lowest_T", float(temp_table["entropy_bits"].iloc[0]))
    ck.record("entropy_highest_T", float(temp_table["entropy_bits"].iloc[-1]))

## A.3 Top-k and top-p: which tokens survive

Temperature only reshapes the distribution. The next two settings **remove** tokens before sampling, so the model can never pick them.
- **Top-k** keeps the `k` most likely tokens and drops the rest. Chapter 2 (**"Top-k"**) presents it mainly as a way to reduce the softmax computation without giving up too much response diversity. It also cuts off the long tail of very unlikely tokens (the many thousands of tokens that each have a tiny probability).
- **Top-p** (nucleus sampling) keeps the *smallest* set of most-likely tokens whose probabilities add up to at least `p`. Unlike top-k, the number of tokens kept adapts: when the model is confident, very few tokens survive; when it is unsure, many do (Chapter 2, **"Sampling Strategies"**).

To see the difference, you'll compare two prompts: the open question from A.1 (many good answers) and a factual question with one obvious answer (`FACT_PROMPT`).

**TODO A.3 (predict first):** with `top_p = 0.9`, which prompt will keep *more* tokens: the open question or the fact question? Write `"open"` or `"fact"`.

In [ ]:
prediction_top_p = ""   # TODO: "open" or "fact"

In [ ]:
FACT_PROMPT = [{"role": "system", "content": "You are Larkspur Ridge IT's help-desk assistant."},
               {"role": "user", "content": "Our office is in Lenexa. Which US state is Lenexa in? Reply with just the state name."}]
fact_logits = next_logits(FACT_PROMPT)
show_top(fact_logits, k=5, title="Fact question, top candidates at temperature 1:")

FILTERS = [{"top_k": 20}, {"top_p": 0.6}, {"top_p": 0.9}]
filter_table = pd.concat([compare_knobs(logits, FILTERS).assign(prompt="open"),
                          compare_knobs(fact_logits, FILTERS).assign(prompt="fact")], ignore_index=True)
display(filter_table[["prompt", "top_k", "top_p", "survivors", "top1", "top1_prob"]])
show_knobs(logits, top_p=0.9)

**Reading the table:** one row per prompt and filter (`top_k=0` and `top_p=1.0` mean that filter is off). `survivors` is how many tokens are still allowed. Top-k keeps the same number for both prompts; top-p keeps a number that depends on how confident the model is. `top1_prob` is the top token's probability *after* filtering: the survivors' probabilities are rescaled so they add up to 1 again, which is why it is higher than in A.1. The chart shows top-p 0.9 on the open question: blue bars only for the survivors.

**TODO A.3 (read off):** copy three survivor counts from the table.

In [ ]:
survivors = {
    "open_top_p_0.6": None,   # TODO: survivors for the open question at top_p 0.6
    "open_top_p_0.9": None,   # TODO: survivors for the open question at top_p 0.9
    "fact_top_p_0.9": None,   # TODO: survivors for the fact question at top_p 0.9
}

In [ ]:
def _n_survive(lg, **s):
    return int(torch.isfinite(apply_knobs(lg, **s)).sum())


_true_surv = {"open_top_p_0.6": _n_survive(logits, top_p=0.6), "open_top_p_0.9": _n_survive(logits, top_p=0.9),
              "fact_top_p_0.9": _n_survive(fact_logits, top_p=0.9)}
try:
    _surv_ok = all(survivors.get(k) is not None and int(survivors[k]) == v for k, v in _true_surv.items())
except (TypeError, ValueError):
    _surv_ok = False
ck.check("A3_survivors", _surv_ok, 2, "Copy the survivors column for those three rows of the table (whole numbers).")
ck.check("A_predictions", str(prediction_temperature).strip().lower() in ("higher", "lower")
         and str(prediction_top_p).strip().lower() in ("open", "fact"), 1,
         "Fill in prediction_temperature (A.2) and prediction_top_p (A.3). Any honest guess earns the point.")
for k, v in _true_surv.items():
    ck.record(f"survivors_{k}", v)
ck.record("prediction_temperature", str(prediction_temperature))
ck.record("prediction_top_p", str(prediction_top_p));

## A.4 Turn the knobs on real answers

So far you looked at one next-token distribution. When a model writes a whole answer, the same settings are applied at **every** token. `model.generate()` (the `transformers` function that writes text) takes them as arguments: `do_sample=True` turns sampling on (without it, `generate()` is greedy), and `temperature`, `top_k`, and `top_p` are the knobs from A.2 and A.3. The course cluster and most commercial model APIs accept the same names.

The test prompt is a job Larkspur really has: a short client announcement about the Thanksgiving office closure.

**TODO A.4:**
1. In `try_settings`, write the `model.generate(...)` call. Pass the `inputs` (unpacked with `**`), turn sampling on, and pass the three settings, plus `num_return_sequences=n` (several answers in one call), `max_new_tokens=max_new_tokens`, and `pad_token_id=tokenizer.eos_token_id` (silences a warning).
2. Fill in `MY_SETTINGS` with at least three settings you want to compare, with three different temperatures (for example a cautious one, a middle one, and a wild one). With sampling on, temperatures must be above 0.

In [ ]:
ANNOUNCE = [{"role": "system", "content": "You are Larkspur Ridge IT's communications assistant."},
            {"role": "user", "content": "Write a friendly two-sentence announcement telling our clients that the "
                                        "Larkspur office is closed on Thanksgiving Day and the Friday after."}]


def try_settings(messages, temperature, top_k, top_p, n=3, max_new_tokens=60):
    """Generate n answers with the given sampling settings. Returns a list of n strings."""
    inputs = chat_inputs(messages)
    L = inputs["input_ids"].shape[1]
    with torch.no_grad():
        # TODO: call model.generate(...) with **inputs, sampling turned on, the three settings,
        #       num_return_sequences=n, max_new_tokens, and pad_token_id=tokenizer.eos_token_id
        out = None
    if out is None:
        return []
    return [tokenizer.decode(o[L:], skip_special_tokens=True).strip() for o in out]


MY_SETTINGS = [
    # TODO: at least three dicts shaped like {"temperature": ..., "top_k": ..., "top_p": ...}, different temperatures
]

In [ ]:
gen_results = {}
if not MY_SETTINGS:
    print("Fill in MY_SETTINGS in the cell above first.")
for s in MY_SETTINGS:
    key = f"T={s.get('temperature')}, top_k={s.get('top_k')}, top_p={s.get('top_p')}"
    try:
        texts = try_settings(ANNOUNCE, s.get("temperature", 1.0), s.get("top_k", 0), s.get("top_p", 1.0))
    except Exception as e:   # e.g. temperature 0 with sampling on
        print(f"{key} -> error: {e}")
        continue
    if not texts:
        print("try_settings returned nothing: finish the TODO in the cell above.")
        break
    gen_results[key] = texts
    print(f"\n=== {key}")
    for t in texts:
        print("  -", t[:220])

**Reading the output:** three announcements per setting. Compare them within a setting (do the three differ?) and across settings (wording, tone, and whether the text stays sensible at your highest temperature).

**TODO A.4 (continued):** describe what you saw in one or two sentences.

In [ ]:
generate_notes = ""   # TODO: what changed in the announcements as you raised the temperature?

In [ ]:
try:
    _probe = try_settings(ANNOUNCE, 0.7, 20, 0.8, n=2, max_new_tokens=8)
except Exception as e:   # a wrong generate(...) call should fail this check, not stop Run all
    print("try_settings raised an error:", type(e).__name__, e)
    _probe = []
ck.check("A4_generate", len(_probe) == 2 and all(isinstance(t, str) for t in _probe), 2,
         "try_settings must return n strings. Check the model.generate(...) call: sampling on, your three settings, "
         "and num_return_sequences=n.")
ck.check("A4_settings", len(gen_results) >= 3 and len({s.get("temperature") for s in MY_SETTINGS}) >= 3, 1,
         "Run at least three settings with three different temperatures (all above 0) in MY_SETTINGS.")
ck.check("A4_notes", len(str(generate_notes).strip()) >= 20, 1,
         "Write a sentence or two in generate_notes about how the outputs changed.")
ck.record("my_settings", [str(s) for s in MY_SETTINGS]);

## A.5 Choose the settings you would ship

This is the decision you'll make at work whenever you set up an AI tool: the same model needs different settings for different jobs. Larkspur needs two:
1. **`helpdesk_policy`:** the help-desk assistant answers policy questions such as *"What is the response target for a P3 ticket for a Premier client?"* The answer comes from a document, and every technician who asks should get the same answer.
2. **`client_announcement`:** drafting a friendly client announcement like the Thanksgiving notice in A.4. Some variety and warmth helps, and a person edits the draft before it goes out.

For reference, Qwen (the model's publisher) suggests `temperature=0.7, top_p=0.8, top_k=20` as a general default for this model in non-thinking mode. You don't have to copy it.

**TODO A.5:** fill in your recommended settings for each job and a one-sentence reason, based on what you saw in A.2-A.4. `temperature` may be `0` for greedy (the next cell then turns sampling off); `top_k=0` and `top_p=1.0` mean that filter is off. The cell after it runs each recommendation three times so you can judge the result; adjust and re-run if you don't like it. Analysis Q2 asks you to defend these choices.

In [ ]:
recommended = {
    "helpdesk_policy":     {"temperature": None, "top_k": None, "top_p": None, "why": ""},   # TODO
    "client_announcement": {"temperature": None, "top_k": None, "top_p": None, "why": ""},   # TODO
}

In [ ]:
SLA_EXCERPT = """Larkspur Ridge IT - response targets (Exhibit B, effective 2026-01-01)
| Priority | Essentials | Professional | Premier |
| P3 | 4 business hours | 4 business hours | 2 business hours |"""
POLICY_Q = [{"role": "system", "content": "Answer using only this document:\n" + SLA_EXCERPT},
            {"role": "user", "content": "What is the response target for a P3 ticket for a Premier client?"}]


def run_recommended(messages, s, n=3):
    t = s.get("temperature")
    if t is None:
        return ["(fill in recommended above first)"]
    if t == 0:   # greedy: sampling off, the same answer every time, so generate once
        inputs = chat_inputs(messages)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        return [tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()] * n
    texts = try_settings(messages, t, s.get("top_k") or 0, s.get("top_p") or 1.0, n=n)
    return texts or ["(finish try_settings in A.4 first)"]


for job, msgs in [("helpdesk_policy", POLICY_Q), ("client_announcement", ANNOUNCE)]:
    print(f"\n=== {job}: {recommended[job]}")
    try:
        for t in run_recommended(msgs, recommended[job]):
            print("  -", t[:220])
    except Exception as e:
        print("  error:", e)

In [ ]:
def _valid_setting(s):
    try:
        return (0 <= float(s["temperature"]) <= 2 and 0 < float(s["top_p"]) <= 1 and int(s["top_k"]) >= 0
                and len(str(s.get("why", "")).strip()) >= 20)
    except (TypeError, ValueError, KeyError):
        return False


_h, _a = recommended.get("helpdesk_policy", {}), recommended.get("client_announcement", {})
_both = _valid_setting(_h) and _valid_setting(_a)
ck.check("A5_settings_valid", _both, 1,
         "Each job needs a temperature from 0 to 2, top_p above 0 and at most 1, top_k of 0 or more, "
         "and a one-sentence reason in why.")
ck.check("A5_fits_the_job", _both and float(_h["temperature"]) < float(_a["temperature"]), 2,
         "Look back at A.2 and A.4: which temperatures gave repeatable answers and which gave variety? "
         "Match each to the job that needs it.")
if _both:
    for job, s in recommended.items():
        ck.record(f"recommended_{job}", {k: s[k] for k in ("temperature", "top_k", "top_p")})

# Part B - Applied lab: what sampling does to Larkspur answers

The helper below samples `n` answers to the same prompt in one **batched** call (`num_return_sequences`: the answers are generated together, which is faster than one at a time). `temperature=0` means greedy, which gives the same output every time, so it runs once and repeats the result. We set `top_k=0, top_p=1.0` so that **only temperature** changes between runs.

In [ ]:
def sample_n(messages, n, temperature, max_new_tokens=40, top_k=0, top_p=1.0, mdl=None, tok=None):
    mdl, tok = mdl or model, tok or tokenizer
    inputs = chat_inputs(messages, tok, mdl)
    L = inputs["input_ids"].shape[1]
    with torch.no_grad():
        if temperature == 0:
            out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                               pad_token_id=tok.eos_token_id)
            return [tok.decode(out[0][L:], skip_special_tokens=True).strip()] * n
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                           top_k=top_k, top_p=top_p, num_return_sequences=n, pad_token_id=tok.eos_token_id)
    return [tok.decode(o[L:], skip_special_tokens=True).strip() for o in out]


print(sample_n(PROMPT, 3, 0.7, max_new_tokens=8))

## B.1 Diversity versus temperature

A common way to measure output diversity is **distinct-n**: the number of *unique* n-word sequences divided by the *total* number of n-word sequences across all samples. (An n-word sequence is called an **n-gram**: with n = 2, "reset the password" contains "reset the" and "the password".) 1.0 means no repetition across samples; values near 0 mean the samples say the same thing. `distinct_n` is provided below.

**TODO B.1 (predict first):** before you run the experiment, guess distinct-2 for 8 answers at `T=0` (greedy) and at `T=1.5`, as numbers from 0 to 1. Write them in `predicted_distinct2`. Any honest guess earns the point.

In [ ]:
# Provided: distinct-n (unique n-grams divided by total n-grams, across all texts)
def distinct_n(texts, n=2):
    """Unique n-grams / total n-grams across all texts (lowercased, split on spaces)."""
    ngrams = []
    for t in texts:
        words = t.lower().split()
        ngrams += [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]   # every run of n words
    return len(set(ngrams)) / len(ngrams) if ngrams else 0.0


print(distinct_n(["a b c", "a b d"], 2), "(3 unique word pairs out of 4)")

# TODO (before you run the next cell): your guesses, numbers from 0 to 1
predicted_distinct2 = {"T=0": None, "T=1.5": None}

In [ ]:
CREATIVE = [{"role": "user", "content": "Write a one-sentence welcome message for a new Larkspur Ridge IT client."}]
N = 8
div_rows, samples_by_t = [], {}
for t in [0, 0.7, 1.0, 1.5]:
    texts = sample_n(CREATIVE, N, t, max_new_tokens=40)
    samples_by_t[t] = texts
    div_rows.append({"T": t, "distinct_1": distinct_n(texts, 1), "distinct_2": distinct_n(texts, 2),
                     "unique_outputs": len(set(texts))})
    print(f"\n=== T={t}")
    for s in texts[:3]:
        print("  -", s[:150])
diversity = pd.DataFrame(div_rows)
diversity

**Reading the table:** one row per temperature `T` (0 is greedy). `distinct_1` and `distinct_2` are distinct-n for single words and for word pairs across the 8 samples; `unique_outputs` is how many of the 8 answers were not exact copies of another. Above the table you see three sample answers for each temperature. Compare the `distinct_2` column with your `predicted_distinct2`.

In [ ]:
def _is_frac(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool) and 0 <= x <= 1


ck.check("B1_prediction", all(_is_frac(predicted_distinct2.get(k)) for k in ("T=0", "T=1.5")), 1,
         "Write both guesses in predicted_distinct2 as numbers from 0 to 1. Any honest guess earns the point.")
d2 = diversity["distinct_2"]
ck.check("B1_diversity_grows", d2.notna().all() and d2.iloc[-1] > d2.iloc[0], 2,
         "distinct-2 at T=1.5 should be higher than at T=0 (greedy repeats one answer). Re-run the experiment cell.")
if d2.notna().all():
    for t, v in zip(diversity["T"], d2):
        ck.record(f"distinct2_T{t}", float(v))
ck.record("predicted_distinct2", {k: v for k, v in predicted_distinct2.items()})

## B.2 Consistency and hallucination on Larkspur facts

Two questions, each sampled 10 times at four temperatures (top-k and top-p off):

1. **Grounded:** the answer is in the context we provide (Larkspur's SLA table). There is one right answer: *2 business hours*.
2. **Ungrounded:** we ask for Larkspur's support phone number and give *no* context. Larkspur is fictional, so the model cannot know it. Any specific number it gives is a **hallucination**. (If your model refuses or says it doesn't know instead, that's a result too: report it in Q2 and connect it to post-training, which you'll see in B.3.)

**TODO B.2 (predict first):** before you run the next cell, decide: at the lowest temperature (0.3), will the model stop inventing phone numbers? Write `"yes"` or `"no"` in `prediction_low_T_stops_hallucination` at the top of the cell. Any honest guess earns the point. The cell then computes, for each temperature, the grounded accuracy (the fraction of the 10 grounded answers that are correct) and the number of different phone numbers among the 10 ungrounded answers.

In [ ]:
prediction_low_T_stops_hallucination = ""   # TODO (before you run this cell): "yes" or "no"

SLA_TABLE = """Larkspur Ridge IT - response targets (Exhibit B, effective 2026-01-01)
| Priority | Essentials | Professional | Premier |
| P1 | 1 hour (business hours only) | 30 minutes (24x7) | 15 minutes (24x7) |
| P2 | 2 business hours | 1 business hour | 1 hour (24x7) |
| P3 | 4 business hours | 4 business hours | 2 business hours |
| P4 | 1 business day | 1 business day | 1 business day |"""
GROUNDED = [{"role": "system", "content": "Answer using only this document:\n" + SLA_TABLE},
            {"role": "user", "content": "What is the response target for a P2 ticket for an Essentials client? "
                                        "Answer in five words or fewer."}]
UNGROUNDED = [{"role": "user", "content": "What is the phone number of Larkspur Ridge IT's support line? "
                                          "Reply with only the phone number."}]


def is_correct_grounded(answer):
    return re.search(r"\b(2|two)\s+business\s+hours?\b", answer.lower()) is not None


def normalize_phone(answer):
    digits = re.sub(r"\D", "", answer)
    return digits if digits else answer.strip().lower()[:40]


cons_rows, examples = [], {}
for t in [0.3, 0.7, 1.0, 1.3]:
    g = sample_n(GROUNDED, 10, t, max_new_tokens=16)
    u = sample_n(UNGROUNDED, 10, t, max_new_tokens=20)
    examples[t] = (g[:3], u[:5])
    # Provided: fraction of grounded answers that are correct, and number of different phone numbers
    grounded_accuracy = sum(is_correct_grounded(a) for a in g) / len(g)
    ungrounded_distinct = len({normalize_phone(a) for a in u})
    cons_rows.append({"T": t, "grounded_accuracy": grounded_accuracy, "ungrounded_distinct": ungrounded_distinct})
consistency = pd.DataFrame(cons_rows)
for t, (g3, u5) in examples.items():
    print(f"T={t}  grounded: {g3}\n        ungrounded: {u5}")
consistency

**Reading the table:** one row per temperature `T`. `grounded_accuracy` is the fraction of the 10 grounded answers that said 2 business hours (1.0 = all correct). `ungrounded_distinct` is how many *different* phone numbers appeared among the 10 ungrounded answers (1 = the model always invented the same one). The printed lines above it show a few raw answers.

In [ ]:
ga, ud = consistency["grounded_accuracy"], consistency["ungrounded_distinct"]
ck.check("B2_prediction", str(prediction_low_T_stops_hallucination).strip().lower() in ("yes", "no"), 1,
         "Write \"yes\" or \"no\" in prediction_low_T_stops_hallucination at the top of the B.2 cell. Any honest guess earns the point.")
ck.check("B2_consistency_table", len(consistency) == 4 and ga.notna().all() and ga.between(0, 1).all()
         and ud.notna().all() and (ud >= 1).all(), 1,
         "The table needs four rows with values in both columns. Re-run the B.2 cell (it samples 80 answers).")
if ga.notna().all() and ud.notna().all():
    ck.record("grounded_acc_T0.3", float(ga.iloc[0]))
    ck.record("grounded_acc_T1.3", float(ga.iloc[-1]))
    ck.record("ungrounded_distinct_T0.3", int(ud.iloc[0]))
    ck.record("ungrounded_distinct_T1.3", int(ud.iloc[-1]))
ck.record("prediction_low_T_stops_hallucination", str(prediction_low_T_stops_hallucination))

## B.3 Base model versus post-trained model

Chapter 2 (**"Post-Training"**) describes two steps that turn a pretrained model into a chat assistant: **supervised finetuning (SFT)** on demonstration conversations, then **preference finetuning** (RLHF, DPO and similar) to make answers more helpful and safe. The base model only learned to continue text. Load the base model and give both models the same three requests: the base model gets the plain text, the instruct model gets it through its chat template (the special-token format that marks the user's turn).

In [ ]:
BASE_ID = "Qwen/Qwen3-1.7B-Base"
base_tok = AutoTokenizer.from_pretrained(BASE_ID)
base_model = AutoModelForCausalLM.from_pretrained(BASE_ID, dtype=DTYPE, device_map="auto")
base_model.eval()
if gpu:
    print(f"GPU memory in use with both models: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


def base_continue(text, max_new_tokens=80):
    enc = base_tok(text, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=base_tok.eos_token_id)
    return base_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


REQUESTS = ["How do I reset my Windows password?",
            "Write a two-sentence apology to a client for missing a response deadline.",
            "What is 17 * 23?"]
pairs = []
for r in REQUESTS:
    b = base_continue(r + "\n")
    i = sample_n([{"role": "user", "content": r}], 1, 0, max_new_tokens=80)[0]
    pairs.append((r, b, i))
    print(f"##### {r}\n--- BASE (plain text continuation):\n{b}\n--- INSTRUCT (chat template):\n{i}\n")

Post-training may also change the *shape* of the distribution: a post-trained model is often more confident (lower entropy) about what comes next. This is the lab's claim to test, not a statement from the chapter, and with only three prompts **your numbers can go either way: report what you see.** We measure both models on the same plain-text prompts; the cell also prints (not graded) the instruct model's entropy when the prompt goes through its chat template, the way it is normally used.

**TODO B.3 (predict first):** which model do you expect to be more confident (lower mean entropy) on these three requests: `"base"` or `"instruct"`? Write it in `prediction_entropy` at the top of the next cell, then run it. The cell computes each model's next-token entropy with `entropy_bits` from A.2 and prints the three entropies (one per request, in the order of `REQUESTS`) and their mean.

In [ ]:
prediction_entropy = ""   # TODO (before you run this cell): "base" or "instruct"


def raw_next_logits(mdl, tok, text):
    enc = tok(text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        return mdl(**enc).logits[0, -1, :].float()


base_entropies = [entropy_bits(raw_next_logits(base_model, base_tok, r + "\n")) for r in REQUESTS]
instruct_entropies = [entropy_bits(raw_next_logits(model, tokenizer, r + "\n")) for r in REQUESTS]

# Provided: the mean of each model's three entropies
mean_entropy_base = sum(base_entropies) / len(base_entropies)
mean_entropy_instruct = sum(instruct_entropies) / len(instruct_entropies)
print("base:", [round(x, 2) for x in base_entropies], "mean", mean_entropy_base)
print("instruct:", [round(x, 2) for x in instruct_entropies], "mean", mean_entropy_instruct)
instruct_chat_entropies = [entropy_bits(next_logits([{"role": "user", "content": r}])) for r in REQUESTS]
print("instruct, through its chat template (not graded):", [round(x, 2) for x in instruct_chat_entropies])

**TODO B.3 (continued):** in one or two sentences each, describe what you observed. These short notes feed analysis Q3.

In [ ]:
observations = {
    "format": "",        # TODO: how did the base model's output format differ from the instruct model's?
    "instructions": "",  # TODO: did the base model follow the request (e.g., two sentences)? did the instruct model?
    "math": "",          # TODO: who got 17 * 23 = 391 right, and how did each present it?
}

In [ ]:
ok_ent = mean_entropy_base is not None and mean_entropy_instruct is not None
ck.check("B3_prediction", str(prediction_entropy).strip().lower() in ("base", "instruct"), 1,
         "Write \"base\" or \"instruct\" in prediction_entropy at the top of the entropy cell. Any honest guess earns the point.")
ck.check("B3_observations", all(len(observations.get(k, "").strip()) >= 20 for k in ["format", "instructions", "math"]), 2,
         "Write at least a full sentence for each of the three observations.")
if ok_ent:
    ck.record("mean_entropy_base", float(mean_entropy_base))
    ck.record("mean_entropy_instruct", float(mean_entropy_instruct))
ck.record("prediction_entropy", str(prediction_entropy))

# Written analysis

Answer in the box under each question (3-6 sentences; Q2 may need up to 8). Use **your own numbers**, and cite Chapter 2 by section name. Q1 is **optional (not graded)**; Q2 and Q3 here plus Q4-Q6 in `lab03_cluster.ipynb` are the graded questions, and the AI-use reflection is in the cluster notebook too.

**Q1 (optional, not graded). Temperature, top-k, top-p.** Using your A.2 table and A.3 survivor counts, explain what temperature does to the distribution and how top-k and top-p differ (how many tokens did each keep for the open question and for the fact question?). Cite Chapter 2's **"Sampling Strategies"**. Qwen recommends temperature 0.7 with top-p 0.8 and top-k 20 for this model in non-thinking mode: why combine them instead of using temperature alone?

*Your answer:*

**Q2. Inconsistency, hallucination, and the settings you'd ship.** Report your B.2 table: grounded accuracy and distinct ungrounded answers at each temperature. Using Chapter 2's **"The Probabilistic Nature of AI"** (inconsistency and hallucination), explain why the model gave confident, *different* phone numbers when it had no context, and why grounding helped. Does lowering the temperature fix hallucination, or only hide it? Then give the settings you chose in A.5 for Larkspur's help-desk assistant and for client announcements, and justify each with your own numbers from A.2-A.4 and B.2 and Chapter 2's **"Sampling Strategies"**. What does the help-desk assistant need besides good sampling settings to avoid made-up answers?

*Your answer:*

**Q3. What post-training adds.** Using your B.3 outputs, observations, and mean entropies, describe how the base and instruct models differed. Which differences come from **supervised finetuning** and which from **preference finetuning**, according to Chapter 2's **"Post-Training"** section? (Reminder: the chapter says SFT trains on high-quality instruction data so the model answers conversations instead of just continuing text, and preference finetuning further shapes responses to match what people prefer. Of your three requests, the apology is the one most likely to show preference finetuning: tone and politeness.) Finally, from your own entropy numbers (this part is the lab's idea, not the chapter's): if the instruct model was more confident, why could that be useful for Larkspur's assistant and a cost for creative or diverse outputs? If it wasn't, say so and suggest why.

*Your answer:*

## Save your progress
Run this last, save (**File > Download > Download .ipynb**), and upload the .ipynb to Canvas together with `lab03_cluster.ipynb`.

In [ ]:
ck.report()